# Legacy Workflow Data Migration & Validation

In [3]:
import pandas as pd

df = pd.read_csv(r"C:\Users\georg\Downloads\Project 1.csv") 

print(df.head())

  Job_ID  Priority_Class  Job_Size_MI  CPU_Required  RAM_Required_GB  \
0  J1000               2         1443            10            24.23   
1  J1001               3         1102             8            58.50   
2  J1002               2          362             4             3.30   
3  J1003               2         1443             2            21.58   
4  J1004               1          469            12             5.53   

   Estimated_Time_Sec  Deadline_Sec Protocol_Sensitivity  Arrival_Time  \
0              290.43        490.35                  Low         55.66   
1              112.01        378.83               Medium        106.06   
2              207.99        112.34                  Low        236.55   
3              426.28        355.93               Medium        484.31   
4              332.78        439.48                  Low        385.90   

  Cluster_ID  Target_Energy_Class  
0         C3                    2  
1         C1                    1  
2         C2  

In [4]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 2000
Columns: 11


In [5]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Job_ID                2000 non-null   object 
 1   Priority_Class        2000 non-null   int64  
 2   Job_Size_MI           2000 non-null   int64  
 3   CPU_Required          2000 non-null   int64  
 4   RAM_Required_GB       2000 non-null   float64
 5   Estimated_Time_Sec    2000 non-null   float64
 6   Deadline_Sec          2000 non-null   float64
 7   Protocol_Sensitivity  2000 non-null   object 
 8   Arrival_Time          2000 non-null   float64
 9   Cluster_ID            2000 non-null   object 
 10  Target_Energy_Class   2000 non-null   int64  
dtypes: float64(4), int64(4), object(3)
memory usage: 172.0+ KB
None


In [9]:
# For every column, tell me how many missing values there are.
print(df.isnull().sum())

Job_ID                  0
Priority_Class          0
Job_Size_MI             0
CPU_Required            0
RAM_Required_GB         0
Estimated_Time_Sec      0
Deadline_Sec            0
Protocol_Sensitivity    0
Arrival_Time            0
Cluster_ID              0
Target_Energy_Class     0
dtype: int64


In [7]:
print("Duplicate Job IDs:", df["Job_ID"].duplicated().sum())

Duplicate Job IDs: 0


In [8]:
print(df["Priority_Class"].value_counts())
print(df["Protocol_Sensitivity"].value_counts())
print(df["Cluster_ID"].value_counts())
print(df["Target_Energy_Class"].value_counts())

Priority_Class
2    982
1    615
3    403
Name: count, dtype: int64
Protocol_Sensitivity
Medium    983
Low       537
High      480
Name: count, dtype: int64
Cluster_ID
C4    528
C3    502
C1    492
C2    478
Name: count, dtype: int64
Target_Energy_Class
1    1017
0     570
2     413
Name: count, dtype: int64


In [10]:
#inspect numerical changes
df.describe()

,Priority_Class,Job_Size_MI,CPU_Required,RAM_Required_GB,Estimated_Time_Sec,Deadline_Sec,Arrival_Time,Target_Energy_Class
count,2000.0000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,1.8940,1118.862500,8.242500,32.646440,259.573940,325.957655,246.873835,0.921500
std,0.7057,523.532643,4.329326,17.924417,134.536943,152.793211,146.027077,0.696836
min,1.0000,200.000000,1.000000,2.010000,30.110000,60.030000,0.600000,0.000000
25%,1.0000,656.000000,4.000000,17.017500,144.160000,196.187500,117.500000,0.000000
50%,2.0000,1123.000000,8.000000,32.100000,256.725000,326.970000,243.690000,1.000000
75%,2.0000,1589.250000,12.000000,48.140000,373.360000,455.222500,376.040000,1.000000
max,3.0000,1999.000000,15.000000,63.970000,499.630000,599.730000,499.420000,2.000000


In [11]:
print("Priority:", sorted(df["Priority_Class"].unique()))
print("CPU:", sorted(df["CPU_Required"].unique()))
print("Clusters:", sorted(df["Cluster_ID"].unique()))
print("Sensitivity:", df["Protocol_Sensitivity"].unique())
print("Energy classes:", sorted(df["Target_Energy_Class"].unique()))

Priority: [np.int64(1), np.int64(2), np.int64(3)]
CPU: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]
Clusters: ['C1', 'C2', 'C3', 'C4']
Sensitivity: ['Low' 'Medium' 'High']
Energy classes: [np.int64(0), np.int64(1), np.int64(2)]


In [12]:
df["Estimated_Time_Sec"] > df["Deadline_Sec"]

0       False
1       False
2        True
3        True
4       False
        ...  
1995    False
1996     True
1997     True
1998    False
1999     True
Length: 2000, dtype: bool

In [13]:
invalid_deadlines = (
    df["Estimated_Time_Sec"] > df["Deadline_Sec"]
).sum()

print("Jobs where estimated time exceeds deadline:", invalid_deadlines)

Jobs where estimated time exceeds deadline: 761


In [15]:
#inspect 761 records.
deadline_issues = df[
    df["Estimated_Time_Sec"] > df["Deadline_Sec"]
]

deadline_issues.head(10)

,Job_ID,Priority_Class,Job_Size_MI,CPU_Required,RAM_Required_GB,Estimated_Time_Sec,Deadline_Sec,Protocol_Sensitivity,Arrival_Time,Cluster_ID,Target_Energy_Class
2,J1002,2,362,4,3.30,207.99,112.34,Low,236.55,C2,0
3,J1003,2,1443,2,21.58,426.28,355.93,Medium,484.31,C2,1
6,J1006,1,1105,5,45.53,437.41,399.72,Low,166.86,C3,1
7,J1007,3,1623,15,22.54,450.17,142.44,Medium,496.02,C2,0
8,J1008,2,1405,12,43.89,462.30,443.49,Medium,51.42,C4,2
10,J1010,1,1952,2,48.89,285.74,73.19,Medium,427.80,C2,1
11,J1011,3,404,15,30.16,378.04,282.34,Medium,463.63,C3,2
12,J1012,3,781,6,27.56,385.76,239.79,Medium,45.29,C4,0
15,J1015,1,1451,9,9.47,404.79,379.07,Low,52.25,C3,0
20,J1020,2,1074,13,23.04,188.61,65.76,High,388.07,C4,0


In [16]:
deadline_issues[
    [
        "Job_ID",
        "Priority_Class",
        "Estimated_Time_Sec",
        "Deadline_Sec",
        "Arrival_Time",
        "Cluster_ID"
    ]
].head(10)

,Job_ID,Priority_Class,Estimated_Time_Sec,Deadline_Sec,Arrival_Time,Cluster_ID
2,J1002,2,207.99,112.34,236.55,C2
3,J1003,2,426.28,355.93,484.31,C2
6,J1006,1,437.41,399.72,166.86,C3
7,J1007,3,450.17,142.44,496.02,C2
8,J1008,2,462.30,443.49,51.42,C4
10,J1010,1,285.74,73.19,427.80,C2
11,J1011,3,378.04,282.34,463.63,C3
12,J1012,3,385.76,239.79,45.29,C4
15,J1015,1,404.79,379.07,52.25,C3
20,J1020,2,188.61,65.76,388.07,C4


In [17]:
print("Number of deadline issues:", len(deadline_issues))
print("Percentage:", round(len(deadline_issues) / len(df) * 100, 2), "%")

Number of deadline issues: 761
Percentage: 38.05 %


In [25]:
#TRANSFORMATION STAGE
#Create the target transformation
target_df = df.copy()

In [19]:
#create priority mapping
priority_mapping = {
    1: "High",
    2: "Medium",
    3: "Low"
}

target_df["Priority"] = target_df["Priority_Class"].map(priority_mapping)

In [20]:
#cluster mapping
cluster_mapping = {
    "C1": "Cluster-01",
    "C2": "Cluster-02",
    "C3": "Cluster-03",
    "C4": "Cluster-04"
}

target_df["Execution_Cluster"] = target_df["Cluster_ID"].map(cluster_mapping)

In [21]:
#energy mapping
energy_mapping = {
    0: "Standard",
    1: "Efficient",
    2: "High-Efficiency"
}

target_df["Energy_Class"] = target_df["Target_Energy_Class"].map(energy_mapping)

In [47]:
#rename the fields
target_df = target_df.rename(columns={
    "Job_ID": "Workflow_ID",
    "Job_Size_MI": "Workload_Size",
    "CPU_Required": "CPU_Units",
    "RAM_Required_GB": "Memory_GB",
    "Estimated_Time_Sec": "Expected_Duration_Sec",
    "Deadline_Sec": "Deadline_Sec",
    "Protocol_Sensitivity": "Sensitivity_Level",
    "Arrival_Time": "Scheduled_Arrival_Sec"
})

In [24]:
target_df.head()

,Workflow_ID,Workload_Size,CPU_Units,Memory_GB,Expected_Duration_Sec,Deadline_Sec,Sensitivity_Level,Scheduled_Arrival_Sec,Priority,Execution_Cluster,Energy_Class
0,J1000,1443,10,24.23,290.43,490.35,Low,55.66,Medium,Cluster-03,High-Efficiency
1,J1001,1102,8,58.50,112.01,378.83,Medium,106.06,Low,Cluster-01,Efficient
2,J1002,362,4,3.30,207.99,112.34,Low,236.55,Medium,Cluster-02,Standard
3,J1003,1443,2,21.58,426.28,355.93,Medium,484.31,Medium,Cluster-02,Efficient
4,J1004,469,12,5.53,332.78,439.48,Low,385.90,High,Cluster-03,Efficient


In [48]:
#VALIDATION
#validate the transformed data
target_df.isnull().sum()

Workflow_ID              0
Workload_Size            0
CPU_Units                0
Memory_GB                0
Expected_Duration_Sec    0
Deadline_Sec             0
Sensitivity_Level        0
Scheduled_Arrival_Sec    0
Priority                 0
Execution_Cluster        0
Energy_Class             0
dtype: int64

In [41]:
#validate record account
print("Source:", len(df))
print("Target:", len(target_df))

Source: 2000
Target: 2000


In [49]:
#validate IDs
print("Source IDs:", df["Job_ID"].nunique())
print("Target IDs:", target_df["Workflow_ID"].nunique())

Source IDs: 2000
Target IDs: 2000


In [50]:
print(target_df.columns.tolist())

['Workflow_ID', 'Workload_Size', 'CPU_Units', 'Memory_GB', 'Expected_Duration_Sec', 'Deadline_Sec', 'Sensitivity_Level', 'Scheduled_Arrival_Sec', 'Priority', 'Execution_Cluster', 'Energy_Class']


In [51]:
#validate mappings
print("\n--- MAPPING VALIDATION ---")

print("Priority values:")
print(target_df["Priority"].value_counts())

print("\nExecution Cluster values:")
print(target_df["Execution_Cluster"].value_counts())

print("\nEnergy Class values:")
print(target_df["Energy_Class"].value_counts())

print("\nSensitivity values:")
print(target_df["Sensitivity_Level"].value_counts())


--- MAPPING VALIDATION ---
Priority values:
Priority
Medium    982
High      615
Low       403
Name: count, dtype: int64

Execution Cluster values:
Execution_Cluster
Cluster-04    528
Cluster-03    502
Cluster-01    492
Cluster-02    478
Name: count, dtype: int64

Energy Class values:
Energy_Class
Efficient          1017
Standard            570
High-Efficiency     413
Name: count, dtype: int64

Sensitivity values:
Sensitivity_Level
Medium    983
Low       537
High      480
Name: count, dtype: int64


In [52]:
print("\n--- PRIORITY MAPPING CHECK ---")

priority_check = pd.crosstab(
    df["Priority_Class"],
    target_df["Priority"]
)

print(priority_check)


--- PRIORITY MAPPING CHECK ---
Priority        High  Low  Medium
Priority_Class                   
1                615    0       0
2                  0    0     982
3                  0  403       0


In [54]:
#migration summary
print("\n==============================")
print("MIGRATION VALIDATION SUMMARY")
print("==============================")

# Record count
source_records = len(df)
target_records = len(target_df)

# Duplicate IDs
duplicate_target_ids = target_df["Workflow_ID"].duplicated().sum()

# Missing values
missing_target_values = target_df.isnull().sum().sum()

# ID reconciliation
source_ids = set(df["Job_ID"])
target_ids = set(target_df["Workflow_ID"])

missing_in_target = source_ids - target_ids
extra_in_target = target_ids - source_ids

# Business-rule validation
deadline_issues = (
    target_df["Expected_Duration_Sec"] >
    target_df["Deadline_Sec"]
)

print("Source records:", source_records)
print("Target records:", target_records)
print("Duplicate target IDs:", duplicate_target_ids)
print("Missing target values:", missing_target_values)
print("Missing source IDs in target:", len(missing_in_target))
print("Extra target IDs:", len(extra_in_target))
print("Deadline exceptions:", deadline_issues.sum())

print("==============================")


MIGRATION VALIDATION SUMMARY
Source records: 2000
Target records: 2000
Duplicate target IDs: 0
Missing target values: 0
Missing source IDs in target: 0
Extra target IDs: 0
Deadline exceptions: 761
